# Constraint Satisfaction

**Content:**
- CSP intro
- CSP problem representation
  - Variables, domains, and constraints
  - Constraint graph
  - Constraint types
- Techniques
  - Backtracking
  - Forward checking
  - Minimum remaining values
  - Least Restrictive value
  - Arc consistency
  - Min-conflicts
- Homework problem

In [ ]:
from collections import deque
import random

: 

## Main idea

- A CSP asks us to choose a value for every variable without breaking any constraints.
- In route search, the sequence of actions can be part of the answer.
- In a CSP, we usually care about the final assignment, not the order in which it was built.

### Examples
- Ddistribute the lectures in rooms and times so that every lecture 
- Aassign workers to shifts
- Place $n$ non-attacking queens on a square board with a size $n$

## Variables, domains, and constraints

| Component | Meaning | Lecture scheduling example |
|:--|:--|:--|
| Variables $X_1,\ldots,X_n$ | Each variable can take a value | One variable per a lecture |
| Domains $D_i$ | Allowed values for each variable | Time and room for a lecture |
| Constraints | Combinations between variable assignments that are permitted | Lectures with the same tutor should not overlap |

- **Partial assignment**: only some variables have values.
- **Consistent assignment**: none of the constraints involving assigned variables is violated.
- **Solution**: a complete assignment satisfying every constraint.

## Example Map Coloring
Color the map without having two bordering parts which have the same color

![Map coloring of Australian states and territories](images/map_coloring.png)


## Constraint graph

> A node is a variable; an edge means that two variables must satisfy a shared rule.

| Region | Adjacent regions |
|:--|:--|
| WA | NT, SA |
| NT | WA, SA, Q |
| SA | WA, NT, Q, NSW, V |
| Q | NT, SA, NSW |
| NSW | SA, Q, V |
| V | SA, NSW |
| T | None |

## Constraint types

| Type | Example | Representation |
|:--|:--|:--|
| Unary | SA cannot be green | Remove green from SA's domain |
| Binary | WA and NT must differ | An edge between WA and NT |
| Higher-order | Three tasks must use three distinct workers | A constraint over several variables |

- Higher-order rules can be represented with hyperedges or a separate constraint node connected to its variables.
- `AllDifferent` is a common global constraint. Its meaning can be expressed with pairwise inequalities, although specialized propagation can be stronger.
- The general-purpose solvers below handle **binary constraints**. Each neighbor relation must appear in both directions; `allowed(x, a, y, b)` tests a proposed pair.

## Hard rules and soft preferences

- A **hard constraint** must hold: a teacher cannot teach two classes at once.
- A **soft constraint** expresses a preference: avoid early classes when possible.
- Soft constraints add penalties to minimize, turning feasibility into constrained optimization.

Availability is often unary; non-overlap is binary; daily teaching-load limits may involve many classes.

## Backtracking search

> Assign one variable, reject conflicting values, and undo a choice when it leads nowhere.

- Choose an unassigned variable and try values compatible with the current assignment.
- If no value works, return to the previous decision.
- Assignments commute: A then B gives the same result as B then A. Do not explore both orders unnecessarily.

- **Complete**: Yes, for finite domains when every remaining choice is eventually tried
- **Optimal**: Finds a valid assignment; no cost objective is defined here
- **Time complexity**: exponential in $n$; there can be $d^n$ complete assignments, plus constraint-checking work
- **Space complexity**: $O(nd)$ extra: each recursive level retains its legal-value list, plus the assignment and stack
- **Data structure**: assignment dictionary and recursion stack

## Minimum remaining values

> Choose the unassigned variable with the fewest currently legal values.

- **MRV** tackles the most restricted decision first, exposing failure sooner.
- Count values compatible with the current assignment, not just the original domain sizes.
- With SA = red and NT = green, WA has only blue available.

## Least constraining value

> Try the value that removes the fewest options from unassigned neighbors.

- **LCV** orders values; MRV chooses variables.
- Suppose A can take red or blue, while its neighbor B can take red or green. Choosing blue leaves both options for B; red removes one.
- Ordering adds work but may reduce backtracking. This version counts immediate eliminations, without full propagation.

## Forward checking

> After an assignment, remove incompatible values from its unassigned neighbors.

- Setting SA = red removes red from WA, NT, Q, NSW, and V.
- An empty domain means this branch cannot succeed; backtrack immediately.
- It filters from the newly assigned variable, without propagating every consequence between unassigned variables.
- For $k$ unassigned neighbors, filtering takes $O(kd)$ constraint checks. This readable implementation also copies all domains, taking $O(nd)$ time and memory per call.

In [6]:
def forward_check(var, value, assignment, domains, neighbors, allowed):
    reduced = {x: values.copy() for x, values in domains.items()}
    reduced[var] = [value]

    for other in neighbors[var]:
        if other not in assignment:
            reduced[other] = [candidate for candidate in reduced[other]
                              if allowed(var, value, other, candidate)]
            if not reduced[other]:
                return None
    return reduced

reduced = forward_check('SA', 'red', {'SA': 'red'},
                        domains, neighbors, different)
print('WA after SA = red:', reduced['WA'])
print('Original WA domain:', domains['WA'])

WA after SA = red: ['green', 'blue']
Original WA domain: ['red', 'green', 'blue']


## Backtracking with early filtering

> Combine MRV, LCV, and forward checking without discarding valid alternatives.

- MRV chooses the variable, LCV orders its values, and forward checking filters the next branch.
- The search remains complete on finite domains and exponential in the worst case.
- Copied domains restore earlier choices automatically, using up to $O(n^2d)$ extra memory in this implementation.

## Arc consistency

> Every remaining value of X must have a supporting value in Y's domain.

For the directed arc $X\rightarrow Y$:

$$\forall x\in D_X,\quad \exists y\in D_Y:\ C_{XY}(x,y).$$

- If X must differ from Y and $D_Y=\{red\}$, remove red from $D_X$.
- Revise both directions: support for X's values and support for Y's values are separate checks.
- Unlike forward checking, arc consistency can propagate between variables that have not been assigned.

## AC-3

> Recheck arcs whenever pruning may have removed a value's last support.

- Put every directed arc in a queue. Remove unsupported values from the first variable's domain.
- If that domain shrinks, requeue incoming arcs from its other neighbors.
- An empty domain proves failure. Otherwise return the reduced domains, not a guaranteed solution.
- **Time complexity**: $O(ed^3)$ with constant-time binary constraint tests
- **Space complexity**: $O(e+nd)$ for the deduplicated queue and copied domains
- **Data structure**: queue of directed arcs and remaining domains

In [8]:
def ac3(domains, neighbors, allowed):
    reduced = {x: values.copy() for x, values in domains.items()}
    if any(not values for values in reduced.values()):
        return None
    queue = deque((x, y) for x in neighbors for y in neighbors[x])
    queued = set(queue)

    while queue:
        x, y = queue.popleft()
        queued.remove((x, y))
        supported = [a for a in reduced[x]
                     if any(allowed(x, a, y, b) for b in reduced[y])]
        if len(supported) < len(reduced[x]):
            reduced[x] = supported
            if not supported:
                return None
            for z in neighbors[x]:
                if z != y and (z, x) not in queued:
                    queue.append((z, x))
                    queued.add((z, x))
    return reduced

chain = {'A': ['B'], 'B': ['A', 'C'], 'C': ['B']}
chain_domains = {'A': ['red'], 'B': ['red', 'blue'], 'C': ['red', 'blue']}
print('Propagated chain:', ac3(chain_domains, chain, different))

Propagated chain: {'A': ['red'], 'B': ['blue'], 'C': ['red']}


## Min-conflicts

> Start with a complete assignment and repair one conflicted variable at a time.

- Pick a random variable involved in a violated constraint.
- Give it a value with the fewest conflicts; break ties randomly.
- Unlike backtracking, all variables already have values. Equal-conflict moves can cross plateaus.

- **Complete**: No finite-run success guarantee
- **Optimal**: No cost objective; any zero-conflict assignment solves the CSP
- **Time complexity**: $O(tk(n+d))$ for $t$ repairs and maximum neighbor count $k\geq1$ in this version, plus initialization
- **Space complexity**: $O(n+d)$ extra, excluding the input graph and domains
- **Data structure**: full assignment and conflict counts; use a fixed seed for repeatability

## N-queens as a CSP

> Place n queens so that no pair shares a row, column, or diagonal.

- **Variable** $Q_c$: the row of the queen in column $c$.
- **Domain**: rows $0,\ldots,n-1$ for every queen.
- One variable per column already prevents column conflicts.
- For distinct columns $c,j$, require $Q_c\ne Q_j$ and $|Q_c-Q_j|\ne|c-j|$.
- Positive board sizes have solutions except $n=2$ and $n=3$.

For four queens, `[1, 3, 0, 2]` means the queen in column 0 is in row 1, and so on.

## Counting attacks efficiently

> Count occupied rows and diagonals instead of comparing every pair of queens.

| Counter | Index for a queen at (row, col) | Array length |
|:--|:--|:--|
| Rows | $row$ | $n$ |
| Descending diagonals | $row-col+n-1$ | $2n-1$ |
| Ascending diagonals | $row+col$ | $2n-1$ |

- For a queen already on the board, the three counters include that queen three times. Subtract 3 to count its attackers.
- Before scoring replacement rows, temporarily remove that queen from the counters.
- Summing per-queen attacks counts each attacking pair twice.

## Min-conflicts with queen counters

> Move a conflicted queen to a row with the fewest attackers.

- Initialize one queen in each column, then build the three counter arrays.
- Remove the chosen queen, score all rows, select a minimum at random, and update the counters.
- **Time complexity**: $O(n)$ initialization and $O(tn)$ for $t$ repairs; this version scans all queens and all candidate rows per repair
- **Space complexity**: $O(n)$ for positions, counters, and temporary scores
- A failed run means the repair budget was exhausted, not that the board is impossible. Restarts can help; they do not give a finite-run guarantee.

## Compare the tools

| Tool | Main decision or action | Main limitation |
|:--|:--|:--|
| Backtracking | Extend a partial assignment | Exponential search in the worst case |
| MRV | Choose the variable with fewest legal values | Computing legal values adds work |
| LCV | Try the value preserving most neighbor options | Local estimates may miss later conflicts |
| Forward checking | Filter neighbors after an assignment | Does not propagate all remaining constraints |
| AC-3 | Remove values without support across an arc | Arc consistency does not imply satisfiability |
| Min-conflicts | Repair a complete assignment | Can stall or exhaust its repair budget |

MRV and LCV order choices; propagation filters values; search resolves remaining alternatives.

## Homework tasks

- Solve the instance with plain backtracking, then with MRV, LCV, and forward checking.
- Count attempted assignments and backtracks; check that both results satisfy every edge constraint.
- Apply AC-3 before search. Explain which domains change, or why none changes.
- Compare several min-conflicts seeds using the same repair budget. Report failures as well as successes.
- Optional: add a soft penalty for slot 3 and search for a valid schedule with minimum total penalty. The first feasible schedule is not necessarily optimal.